In [0]:
import mlflow
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

mlflow.set_registry_uri("databricks-uc")

date =  dbutils.widgets.get("date")

model_name = "feature_store.upsell.churn"
model_version = 3

model_uri = f"models:/{model_name}/{model_version}"

model_pyfunc = mlflow.pyfunc.load_model(model_uri)
run_id = model_pyfunc.metadata.run_id

model = mlflow.sklearn.load_model(f"runs:/{run_id}/model")

In [0]:
lookups = [
    FeatureLookup(table_name="feature_store.upsell.fs_geral", lookup_key=['IdCliente', 'dtRef']),
    FeatureLookup(table_name="feature_store.upsell.fs_pontos", lookup_key=['IdCliente', 'dtRef']),
    FeatureLookup(table_name="feature_store.upsell.fs_transacoes", lookup_key=['IdCliente', 'dtRef']),
    FeatureLookup(table_name="feature_store.upsell.fs_dia_horario", lookup_key=['IdCliente', 'dtRef']),
]

query = f"""
    SELECT dtRef,
            IdCliente
    FROM feature_store.upsell.fs_geral
    WHERE dtRef = '{date}'
"""

df = spark.sql(query)

fe = FeatureEngineeringClient()

predict_set = fe.create_training_set(df=df, 
                                     feature_lookups=lookups, 
                                     label=None)

df_predict = predict_set.load_df().toPandas()

In [0]:
probas = model.predict_proba(df_predict[model.feature_names_in_])

df_model = df_predict[['dtRef', 'IdCliente']].copy()
df_model['descModelName'] = model_name
df_model['nrModelVersion'] = model_version

columns = ['dtRef', 'descModelName', 'nrModelVersion', "IdCliente"]

df_model[model.classes_] = probas
df_model = df_model.set_index(columns).stack().reset_index()

df_model.columns = columns + ['descLabel', 'nrProbaLabel']

In [0]:
sdf = spark.createDataFrame(df_model)

query_delete ="""
DELETE FROM feature_store.upsell.models 
WHERE dtRef = '{date}' 
AND descModelName = '{model_name}'
"""

spark.sql(query_delete)

(sdf.write
    .format('delta')
    .mode('append')
    .partitionBy(['dtRef', 'descModelName'])
    .saveAsTable('feature_store.upsell.models'))